In [ ]:
'''Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries
: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:  

In [2]:
import pandas as pd

In [4]:
roll_number=1024170061
categories=["billing", "account", "general"]
digits=[4,9]
c1=categories[digits[0]%3]
c2=categories[digits[1]%3]
print("Categories as per my roll number are:",c1,c2)   
fixed_entries = [ {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
                  {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"}, 
                  {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
                  {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
                   ] 
other_entries=[{"question": "how do I update my registered address", "answer": "You can do to Settings>profile>update address", "keywords": "address update account", "category": c1},
            {"question": "how do I check my fee payment status", "answer": "You can do to Settings>billings>Fee payment Status", "keywords": "fee payment status", "category": c2}
             ]
all_entries=fixed_entries+other_entries
df=pd.DataFrame(all_entries)
print(df)


Categories as per my roll number are: account billing
                                question  \
0                 what is the annual fee   
1                  how to reset password   
2            what are your working hours   
3                  how can i pay the fee   
4  how do I update my registered address   
5   how do I check my fee payment status   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4      You can do to Settings>profile>update address  address update account   
5  You can do to Settings>billings>Fee payment St...      fee payment status   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  account 

In [ ]:
'''Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence'''

In [11]:
def score_query(query,df):
    query_words=set(query.lower().split())
    results=[]
    for index,row in df.iterrows():
        keywords=set(row["keywords"].lower().split())
        score=len(query_words.intersection(keywords))
        if(score>0):
            results.append({
                "question":row["question"],
                "answer":row["answer"],
                "category":row["category"],
                "confidence":score})
    results.sort(key=lambda x:x["confidence"],reverse=True)
    return results
query="how can i pay fee"
matches=score_query(query,df)
for match in matches:
    print(match)

{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'confidence': 2}
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'confidence': 1}
{'question': 'how do I check my fee payment status', 'answer': 'You can do to Settings>billings>Fee payment Status', 'category': 'billing', 'confidence': 1}


In [ ]:
'''Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category.
Call it using the category of one of the personalized entries from Q1, and print the result. '''

In [13]:
def same_category(category_name, df):
    return df[df["category"]==category_name]["question"]
result=same_category("billing",df)
print(result)

0                  what is the annual fee
3                   how can i pay the fee
5    how do I check my fee payment status
Name: question, dtype: object


In [ ]:
'''Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword,
add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.''' 

In [15]:
index=1
new_word=input("Enter new keyword")
df.loc[index,"keywords"]=df.loc[index,"keywords"]+" "+new_word
filename="1024170061_faq_data.csv"
df.to_csv(filename,index=False)
print("\nUpdated DataFrame:")
print(df)
print("\n File saved as:",filename)

Enter new keyword forgot



Updated DataFrame:
                                question  \
0                 what is the annual fee   
1                  how to reset password   
2            what are your working hours   
3                  how can i pay the fee   
4  how do I update my registered address   
5   how do I check my fee payment status   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4      You can do to Settings>profile>update address   
5  You can do to Settings>billings>Fee payment St...   

                             keywords category  
0               fee cost price charge  billing  
1  password reset login forgot forgot  account  
2              hours timing open time  general  
3                 pay payment upi fee  billing  
4              address update acc

In [ ]:
'''Q5: Using groupby, print how many FAQ entries you have per category.'''

In [16]:
category_count=df.groupby("category").size()
print(category_count)

category
account    2
billing    3
general    1
dtype: int64


In [ ]:
'''Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score,
it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match.
Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.'''

In [23]:
def score_query_tie(query,df):
    query_words=set(query.lower().split())
    results=[]
    for index,row in df.iterrows():
        keywords=set(row["keywords"].lower().split())
        score=len(query_words & keywords)
        if(score>0):
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category":row["category"],
                "confidence": score})
    results.sort(key=lambda x:x["confidence"],reverse=True)
    if not results:
        print("No matching record found")
        return
    highest_score=results[0]["confidence"]
    highest_matches=[
        result for result in results
        if result["confidence"]==highest_score]
    if(len(highest_matches))>1:
        print("Tie found!All highest scoring matches:")
        for result in highest_matches:
            print(result)
    else:
        print("Best match:")
        print(highest_matches[0])

print("Test 1:Tie")
score_query_tie("fee",df)

print("Test 2:No tie")
score_query_tie("password",df)

Test 1:Tie
Tie found!All highest scoring matches:
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'confidence': 1}
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'confidence': 1}
{'question': 'how do I check my fee payment status', 'answer': 'You can do to Settings>billings>Fee payment Status', 'category': 'billing', 'confidence': 1}
Test 2:No tie
Best match:
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'confidence': 1}
